In [0]:
spark.sql('SHOW TABLES').show()

+--------+--------------------+-----------+
|database|           tableName|isTemporary|
+--------+--------------------+-----------+
| default|      bigmart_silver|      false|
| default|     bronze_customer|      false|
| default|        bronze_order|      false|
| default|   bronze_order_item|      false|
| default|bronze_order_payment|      false|
| default|      bronze_product|      false|
| default|gold_customer_ana...|      false|
| default|    gold_kpi_summary|      false|
| default|gold_outlet_dashb...|      false|
| default|gold_payment_anal...|      false|
| default|gold_product_anal...|      false|
| default|gold_profile_anal...|      false|
| default|gold_sales_by_cit...|      false|
| default|gold_sales_by_ite...|      false|
| default|gold_sales_by_out...|      false|
| default|   gold_top_products|      false|
| default|silver_customer_rank|      false|
| default|silver_customer_s...|      false|
| default| silver_product_rank|      false|
| default|   silver_sales_ecom| 

In [0]:
customer=spark.table('bronze_customer')
order=spark.table('bronze_order')
order_item=spark.table('bronze_order_item')
order_payment=spark.table('bronze_order_payment')
product=spark.table('bronze_product')

## Cheching data quality

In [0]:
from pyspark.sql.functions import col
for c in customer.columns:
    null_count=customer.filter(col(c).isNull()).count()
    print(f"{c} : {null_count}")

customer_id : 0
customer_unique_id : 0
customer_zip_code_prefix : 0
customer_city : 0
customer_state : 0


In [0]:
for c in order.columns:
    null_count=order.filter(col(c).isNull()).count()
    print(f"{c} : {null_count}")

order_id : 0
customer_id : 0
order_status : 0
order_purchase_timestamp : 0
order_approved_at : 160
order_delivered_carrier_date : 1783
order_delivered_customer_date : 2965
order_estimated_delivery_date : 0


In [0]:
for c in order_item.columns:
    null_count=order_item.filter(col(c).isNull()).count()
    print(f"{c} : {null_count}")

order_id : 0
order_item_id : 0
product_id : 0
seller_id : 0
shipping_limit_date : 0
price : 0
freight_value : 0


In [0]:
for c in order_payment.columns:
    null_count=order_payment.filter(col(c).isNull()).count()
    print(f"{c} : {null_count}")

order_id : 0
payment_sequential : 0
payment_type : 0
payment_installments : 0
payment_value : 0


In [0]:
for c in product.columns:
    null_count=product.filter(col(c).isNull()).count()
    print(f"{c} : {null_count}")

product_id : 0
product_category_name : 610
product_name_lenght : 610
product_description_lenght : 610
product_photos_qty : 610
product_weight_g : 2
product_length_cm : 2
product_height_cm : 2
product_width_cm : 2


In [0]:
product=product.fillna({'product_category_name' : 'unknown'
               ,'product_name_lenght' :0
               ,"product_description_lenght":0
               ,"product_photos_qty":0})

In [0]:
product=product.dropna(subset=['product_weight_g','product_length_cm','product_height_cm','product_width_cm'])

In [0]:
for c in product.columns:
    null_count=product.filter(col(c).isNull()).count()
    print(f"{c} : {null_count}")

product_id : 0
product_category_name : 0
product_name_lenght : 0
product_description_lenght : 0
product_photos_qty : 0
product_weight_g : 0
product_length_cm : 0
product_height_cm : 0
product_width_cm : 0


### checking dublicate

In [0]:
customer.count()
customer.dropDuplicates().count()
order.count()
order.dropDuplicates().count()
order_item.count()
order_item.dropDuplicates().count()
order_payment.count()
order_payment.dropDuplicates().count()
product.count()
product.dropDuplicates().count()
print(f'{"order_dub"}:{order.count()-order.dropDuplicates().count()}')
print(f'{"order_item_dub"}:{order_item.count()-order_item.dropDuplicates().count()}')
print(f'{"order_payment_dub"}:{order_payment.count()-order_payment.dropDuplicates().count()}')
print(f'{"customer_dub"}:{customer.count()-customer.dropDuplicates().count()}')
print(f'{"product_dub"}:{product.count()-product.dropDuplicates().count()}')


order_dub:0
order_item_dub:0
order_payment_dub:0
customer_dub:0
product_dub:0


### Primary key

In [0]:
cust_count=customer.select('customer_id').distinct().count()
customer.count()
print(f'{"primary_key_cust"}:{customer.count()-cust_count}')
order_count=order.select('order_id').distinct().count()
order.count()
print(f'primary_key_order:{order.count()-order_count}')
order_item_count=order_item.select('order_id').distinct().distinct().count()
order_item.count()
print(f'{"primary_key_item"}:{order_item.count()-order_item_count}')
order_payment_count=order_payment.select('order_id').distinct().distinct().count()
order_payment.count()
print(f'{"primary_key_payment"}:{order_payment.count()-order_payment_count}')
product_count=product.select('product_id').distinct().distinct().count()
product.count()
print(f'{"primary_key_product"}:{product.count()-product_count}')

primary_key_cust:0
primary_key_order:0
primary_key_item:13984
primary_key_payment:4446
primary_key_product:0


## Data Standarization

In [0]:
from pyspark.sql.functions import initcap, trim
customer=customer.withColumn(('customer_city'),initcap(trim(col('customer_city'))))


In [0]:
display(customer.limit(10))

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,Franca,SP
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,Sao Bernardo Do Campo,SP
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,Sao Paulo,SP
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,Mogi Das Cruzes,SP
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,Campinas,SP
879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,Jaragua Do Sul,SC
fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,4534,Sao Paulo,SP
5e274e7a0c3809e14aba7ad5aae0d407,57b2a98a409812fe9618067b6b8ebe4f,35182,Timoteo,MG
5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,Curitiba,PR
4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,30575,Belo Horizonte,MG


## Multi Table Joins

In [0]:
silver_sales=customer.join(order,customer.customer_id==order.customer_id,'inner')\
                      .join(order_item,order.order_id==order_item.order_id,'inner')\
                      .join(order_payment,order.order_id==order_payment.order_id,'inner')\
                      .join(product,order_item.product_id==product.product_id,'inner')
display(silver_sales.limit(10))


customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,order_id,payment_sequential,payment_type,payment_installments,payment_value,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
5e43c982ccb91984104e2aaaa062e652,a655539d1946324cc7a22fd0085cc384,5374,Sao Paulo,SP,1431ee47e31c27528b8ae56b66689afc,5e43c982ccb91984104e2aaaa062e652,delivered,2017-08-09T18:20:16.000Z,2017-08-11T16:15:13.000Z,2017-08-14T14:35:01.000Z,2017-08-15T22:29:56.000Z,2017-08-24T00:00:00.000Z,1431ee47e31c27528b8ae56b66689afc,1,def59eb2e17b32b980b5341984f6b500,f8db351d8c4c4c22c6835c19a46f01b0,2017-08-17T16:15:13.000Z,22.9,8.27,1431ee47e31c27528b8ae56b66689afc,2,voucher,1,30.91,def59eb2e17b32b980b5341984f6b500,pet_shop,50,365,3,300,24,12,12
3e4442cd2958fbd08dc760884b222ae2,15010a153e79bb2f1fe810486783f5dd,95400,Sao Francisco De Paula,RS,4813c50e3548fa87684c05b730ecccee,3e4442cd2958fbd08dc760884b222ae2,delivered,2018-01-17T16:49:02.000Z,2018-01-18T02:11:14.000Z,2018-01-19T19:38:59.000Z,2018-02-05T21:09:14.000Z,2018-02-23T00:00:00.000Z,4813c50e3548fa87684c05b730ecccee,1,4ebab6a2135a2477b657bda38646b64b,406822777a0b9eb5c50e442dd4cd3ec5,2018-01-24T02:11:14.000Z,25.9,15.1,4813c50e3548fa87684c05b730ecccee,1,boleto,1,41.0,4ebab6a2135a2477b657bda38646b64b,brinquedos,38,853,3,442,20,16,14
4f0fa92e598fc48c2f7771e220791a55,2c6f62969907bc67dad42631fda31a53,23550,Rio De Janeiro,RJ,a4d782cea9ad01e830a3603bc4e7a716,4f0fa92e598fc48c2f7771e220791a55,delivered,2018-03-16T21:34:22.000Z,2018-03-16T21:50:22.000Z,2018-03-21T19:22:09.000Z,2018-04-02T22:38:34.000Z,2018-04-23T00:00:00.000Z,a4d782cea9ad01e830a3603bc4e7a716,1,bee2e070c39f3dd2f6883a17a5f0da45,4e922959ae960d389249c378d1c939f5,2018-03-26T21:50:22.000Z,180.0,17.8,a4d782cea9ad01e830a3603bc4e7a716,1,credit_card,1,197.8,bee2e070c39f3dd2f6883a17a5f0da45,informatica_acessorios,53,871,4,175,20,20,20
96d3328d8b8625bc4d656279e1515bee,e5f4c50b9b79024bcf048ad25fa518b6,23088,Rio De Janeiro,RJ,0ac372f92fbec55ca27ef4fa99a6a13a,96d3328d8b8625bc4d656279e1515bee,delivered,2018-01-11T14:01:09.000Z,2018-01-11T14:12:26.000Z,2018-01-13T02:51:29.000Z,2018-01-24T16:39:08.000Z,2018-02-09T00:00:00.000Z,0ac372f92fbec55ca27ef4fa99a6a13a,1,b60856ce32d90658dbf99b9485327c25,8b321bb669392f5163d04c59e235e066,2018-01-17T14:12:26.000Z,9.0,14.1,0ac372f92fbec55ca27ef4fa99a6a13a,1,credit_card,1,23.1,b60856ce32d90658dbf99b9485327c25,eletronicos,40,429,1,150,25,7,16
84353738a84112675537c08797692415,b6658698e26b7eb7721441be5834e782,74210,Goiania,GO,bb3b9252816ff94fc9f21944cfa0b6d1,84353738a84112675537c08797692415,delivered,2018-05-04T16:17:14.000Z,2018-05-05T16:13:44.000Z,2018-05-09T13:28:00.000Z,2018-05-22T22:48:18.000Z,2018-06-05T00:00:00.000Z,bb3b9252816ff94fc9f21944cfa0b6d1,1,7a10781637204d8d10485c71a6108a2e,4869f7a5dfa277a7dca6462dcf3b52b2,2018-05-10T16:13:44.000Z,199.0,0.0,bb3b9252816ff94fc9f21944cfa0b6d1,1,credit_card,5,199.0,7a10781637204d8d10485c71a6108a2e,relogios_presentes,42,236,1,342,18,13,15
7f10859af8483458fbe59d40da3d04f5,08114c58968d782002f7cbb2d4cdf823,13825,Holambra,SP,aac01d5ce6fab5aa0e191d71aedbaa3c,7f10859af8483458fbe59d40da3d04f5,delivered,2017-01-28T13:24:20.000Z,2017-01-28T14:31:38.000Z,2017-01-31T08:45:22.000Z,2017-02-09T11:03:47.000Z,2017-03-15T00:00:00.000Z,aac01d5ce6fab5aa0e191d71aedbaa3c,1,6fba9843ddfdebe33d2625afc8001eb7,7008613ea464bad5cb9b83456e1e6a8f,2017-02-01T13:24:20.000Z,35.5,18.0,aac01d5ce6fab5aa0e191d71aedbaa3c,1,credit_card,4,47.68,6fba9843ddfdebe33d2625afc8001eb7,esporte_lazer,36,258,2,800,28,6,26
1ba3670c68bda95d41ac806d86976ab0,20262965ae07d2db3b197caaa56a623e,21815,Rio De Janeiro,RJ,567320164aa96555310f5c8ed6e8c5d7,1ba3670c68b

In [0]:
silver_sales.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- order_id: strin

In [0]:
silver_sales.count()

117581

In [0]:
silver_sales = silver_sales.select(
    customer.customer_id,
    customer.customer_unique_id,
    customer.customer_zip_code_prefix,
    customer.customer_city,
    customer.customer_state,

    order.order_id,
    order.order_status,
    order.order_purchase_timestamp,
    order.order_approved_at,
    order.order_delivered_carrier_date,
    order.order_delivered_customer_date,
    order.order_estimated_delivery_date,

    order_item.order_item_id,
    order_item.product_id,
    order_item.seller_id,
    order_item.shipping_limit_date,
    order_item.price,
    order_item.freight_value,

    order_payment.payment_sequential,
    order_payment.payment_type,
    order_payment.payment_installments,
    order_payment.payment_value,

    product.product_category_name,
    product.product_name_lenght,
    product.product_description_lenght,
    product.product_photos_qty,
    product.product_weight_g,
    product.product_length_cm,
    product.product_height_cm,
    product.product_width_cm
)

In [0]:
silver_sales.columns

['customer_id',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state',
 'order_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'order_item_id',
 'product_id',
 'seller_id',
 'shipping_limit_date',
 'price',
 'freight_value',
 'payment_sequential',
 'payment_type',
 'payment_installments',
 'payment_value',
 'product_category_name',
 'product_name_lenght',
 'product_description_lenght',
 'product_photos_qty',
 'product_weight_g',
 'product_length_cm',
 'product_height_cm',
 'product_width_cm']

##  Feature Engineering

In [0]:
from pyspark.sql.functions import year,month,dayofmonth
silver_sales=silver_sales.withColumn('Order_year',year('order_purchase_timestamp'))
silver_sales=silver_sales.withColumn('Order_month',month('order_purchase_timestamp'))
silver_sales=silver_sales.withColumn('Order_day',dayofmonth('order_purchase_timestamp'))

In [0]:
from pyspark.sql.functions import current_timestamp
silver_sales=silver_sales.withColumn('Order_timestamp', current_timestamp())

## Window_Functions

### Customer Rank (by Total Spend)

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank
from pyspark.sql.functions import sum
customer_rank=silver_sales.groupBy('customer_id')\
                           .agg(sum('payment_value').alias('total_spend'))
window_spec=Window.orderBy(col('total_spend').desc())
customer_rank=customer_rank.withColumn('customer_rank',dense_rank().over(window_spec))

display(customer_rank.limit(10))


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


customer_id,total_spend,customer_rank
1617b1357756262bfa56ab541c47bc16,109312.64,1
bd5d39761aa56689a265d95d8d32b8be,45256.00000000001,2
be1b70680b9f9694d8c70f41fa3dc92b,44048.000000000015,3
05455dfa7cd02f13d132aa7a6a9729c6,36489.24,4
1ff773612ab8934db89fd5afa8afe506,30185.999999999993,5
ec5b2ba62e574342386871631fafd3fc,29099.52,6
e7d6802668de6e74d0d6c56565bf2a24,22346.6,7
8c20d9bfbc96c5d39025d77a3ba83d7f,21874.049999999996,8
f7622098214b4634b7fe7eee269b5426,19457.04,9
71901689c5f3e5adc27b1dd16b33f0b8,19174.38,10


## Product Rank (by Revenue)

In [0]:
product_rank=silver_sales.groupBy('product_id','product_category_name')\
                         .agg(sum('payment_value').alias('revenue'))\
                         .withColumn('product_rank',dense_rank().over(Window.orderBy(col('revenue').desc())))

display(product_rank.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


product_id,product_category_name,revenue,product_rank
5769ef0a239114ac3a854af00df129e4,telefonia_fixa,109312.64,1
bb50f2e236e5eea0100680137654686c,beleza_saude,81887.42000000013,2
422879e10f46682990de24d770e7f83d,ferramentas_jardim,79512.22000000002,3
d1c427060a0f73f6b889a5c7c61f2ac4,informatica_acessorios,70557.89999999997,4
6cdd53843498f92890544667809f1595,beleza_saude,64825.66999999995,5
d5991653e037ccb7af6ed7d94246b249,informatica_acessorios,64143.259999999944,6
aca2eb7d00ea1a7b8ebd4e68314663af,moveis_decoracao,63788.120000000265,7
a62e25e09e05e6faf31d90c6ec1aa3d1,relogios_presentes,63167.36999999991,8
99a4788cb24856965c36a24e339b6058,cama_mesa_banho,63161.39999999987,9
3dd2a17168ec895c781a9191c1e95ad7,informatica_acessorios,58962.13999999998,10


## Top Seller Flag

In [0]:
from pyspark.sql.functions import when
seller_revenue=silver_sales.groupBy('seller_id')\
                           .agg(sum('payment_value').alias('revenue'))\
                            .withColumn('seller_rank',dense_rank().over(Window.orderBy(col('revenue').desc())))
seller_revenue=seller_revenue.withColumn('top_seller_flag',when(col('seller_rank')<=10,'y')\
                                           .otherwise('n'))

display(seller_revenue.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


seller_id,revenue,seller_rank,top_seller_flag
7c67e1448b00f6e969d365cea6b010ab,507166.91,1,y
1025f0e2d44d7041d6cf58b6550e0bfa,308222.0400000005,2,y
4a3ca9315b744ce9f8e9374361493884,301245.27000000206,3,y
1f50f920176fa81dab994f9023523100,290253.42000000196,4,y
53243585a1d6dc2643021fd1853d8905,284903.08000000054,5,y
da8622b14eb17ae2831f4ac5b9dab84a,272219.32,6,y
4869f7a5dfa277a7dca6462dcf3b52b2,264166.1200000002,7,y
955fee9216a65b617aa5c0531780ce60,236322.29999999987,8,y
fa1c13f2614d7b5c4749cbc52fecda94,206513.22999999984,9,y
7e93a43ef30c4f03f38b393420bc753a,185134.20999999996,10,y


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


# Customer Segmentation

In [0]:
customer_segmentation=silver_sales.groupBy('customer_id')\
                                  .agg(sum('payment_value').alias('total_spend'))\
                                  .withColumn(('customer_segment')\
                                  ,when(col('total_spend')>5000,'VIP')\
                                  .when(col('total_spend')>1000,'Regular')\
                                  .otherwise('Low Value')
                                  )
display(customer_segmentation.limit(20))


customer_id,total_spend,customer_segment
8ab97904e6daea8866dbdbc4fb7aad2c,28.62,Low Value
295ae9b35379e077273387ff64354b6f,43.0,Low Value
df9b032b2ad0fd6bf37dfb48e5f83845,29.92,Low Value
332df68ccac2f2f7d9e11299188f8bce,120.84,Low Value
0489975a325480c9e385e9f135bb13c3,170.43,Low Value
5883f965ac70043c2e908c3657c5d548,67.5,Low Value
0a11cb0fb65032da800b780afcc1a1b7,180.95,Low Value
a51b373c427132a77b112ca10f502c4f,187.19,Low Value
b7ee9b824b4a83b389379ac70585bf3b,847.34,Low Value
148ecba880f508ca2d5b6744b6a7b6c5,165.4,Low Value


In [0]:
silver_sales.write.format('delta').mode('overwrite').saveAsTable('silver_sales_ecom')

In [0]:
spark.sql("select * from silver_sales_ecom limit 10").display()

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,payment_sequential,payment_type,payment_installments,payment_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,Order_year,Order_month,Order_day,Order_timestamp
5e43c982ccb91984104e2aaaa062e652,a655539d1946324cc7a22fd0085cc384,5374,Sao Paulo,SP,1431ee47e31c27528b8ae56b66689afc,delivered,2017-08-09T18:20:16.000Z,2017-08-11T16:15:13.000Z,2017-08-14T14:35:01.000Z,2017-08-15T22:29:56.000Z,2017-08-24T00:00:00.000Z,1,def59eb2e17b32b980b5341984f6b500,f8db351d8c4c4c22c6835c19a46f01b0,2017-08-17T16:15:13.000Z,22.9,8.27,2,voucher,1,30.91,pet_shop,50,365,3,300,24,12,12,2017,8,9,2026-06-10T04:36:07.285Z
3e4442cd2958fbd08dc760884b222ae2,15010a153e79bb2f1fe810486783f5dd,95400,Sao Francisco De Paula,RS,4813c50e3548fa87684c05b730ecccee,delivered,2018-01-17T16:49:02.000Z,2018-01-18T02:11:14.000Z,2018-01-19T19:38:59.000Z,2018-02-05T21:09:14.000Z,2018-02-23T00:00:00.000Z,1,4ebab6a2135a2477b657bda38646b64b,406822777a0b9eb5c50e442dd4cd3ec5,2018-01-24T02:11:14.000Z,25.9,15.1,1,boleto,1,41.0,brinquedos,38,853,3,442,20,16,14,2018,1,17,2026-06-10T04:36:07.285Z
4f0fa92e598fc48c2f7771e220791a55,2c6f62969907bc67dad42631fda31a53,23550,Rio De Janeiro,RJ,a4d782cea9ad01e830a3603bc4e7a716,delivered,2018-03-16T21:34:22.000Z,2018-03-16T21:50:22.000Z,2018-03-21T19:22:09.000Z,2018-04-02T22:38:34.000Z,2018-04-23T00:00:00.000Z,1,bee2e070c39f3dd2f6883a17a5f0da45,4e922959ae960d389249c378d1c939f5,2018-03-26T21:50:22.000Z,180.0,17.8,1,credit_card,1,197.8,informatica_acessorios,53,871,4,175,20,20,20,2018,3,16,2026-06-10T04:36:07.285Z
96d3328d8b8625bc4d656279e1515bee,e5f4c50b9b79024bcf048ad25fa518b6,23088,Rio De Janeiro,RJ,0ac372f92fbec55ca27ef4fa99a6a13a,delivered,2018-01-11T14:01:09.000Z,2018-01-11T14:12:26.000Z,2018-01-13T02:51:29.000Z,2018-01-24T16:39:08.000Z,2018-02-09T00:00:00.000Z,1,b60856ce32d90658dbf99b9485327c25,8b321bb669392f5163d04c59e235e066,2018-01-17T14:12:26.000Z,9.0,14.1,1,credit_card,1,23.1,eletronicos,40,429,1,150,25,7,16,2018,1,11,2026-06-10T04:36:07.285Z
84353738a84112675537c08797692415,b6658698e26b7eb7721441be5834e782,74210,Goiania,GO,bb3b9252816ff94fc9f21944cfa0b6d1,delivered,2018-05-04T16:17:14.000Z,2018-05-05T16:13:44.000Z,2018-05-09T13:28:00.000Z,2018-05-22T22:48:18.000Z,2018-06-05T00:00:00.000Z,1,7a10781637204d8d10485c71a6108a2e,4869f7a5dfa277a7dca6462dcf3b52b2,2018-05-10T16:13:44.000Z,199.0,0.0,1,credit_card,5,199.0,relogios_presentes,42,236,1,342,18,13,15,2018,5,4,2026-06-10T04:36:07.285Z
7f10859af8483458fbe59d40da3d04f5,08114c58968d782002f7cbb2d4cdf823,13825,Holambra,SP,aac01d5ce6fab5aa0e191d71aedbaa3c,delivered,2017-01-28T13:24:20.000Z,2017-01-28T14:31:38.000Z,2017-01-31T08:45:22.000Z,2017-02-09T11:03:47.000Z,2017-03-15T00:00:00.000Z,1,6fba9843ddfdebe33d2625afc8001eb7,7008613ea464bad5cb9b83456e1e6a8f,2017-02-01T13:24:20.000Z,35.5,18.0,1,credit_card,4,47.68,esporte_lazer,36,258,2,800,28,6,26,2017,1,28,2026-06-10T04:36:07.285Z
1ba3670c68bda95d41ac806d86976ab0,20262965ae07d2db3b197caaa56a623e,21815,Rio De Janeiro,RJ,567320164aa96555310f5c8ed6e8c5d7,delivered,2017-03-04T14:03:59.000Z,2017-03-04T14:15:13.000Z,2017-03-08T07:16:50.000Z,2017-03-14T13:00:30.000Z,2017-03-29T00:00:00.000Z,1,f750e77aeeecafcb47516be7f8f278b8,b7e1750c1157341d2abd0251e07c186b,2017-03-13T14:03:59.000Z,39.9,14.11,1,credit_card,5,54.01,fashion_esporte,29,404,2,400,16,10,11,2017,3,4,2026-06-10T04:36:07.285Z
ec8cce4c98a82dead34ffd130e9c6d85,9579b888e112b9bd13772b6ee70112e4,30644,Belo Horizonte,MG,13ce532ef1a57ec06218af7cb3634241,delivered,2018-02-25T14:17:14.000Z,2018-02-25T14:30:29.000Z,2018-02-26T19:43:48.000Z,2018-03-03T20:18:48.000Z,2018-03-21T00:00:00.0

In [0]:
customer_rank.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_customer_rank")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
product_rank.write\
            .format("delta")\
            .mode("overwrite")\
            .saveAsTable("silver_product_rank")
seller_revenue.write\
            .format("delta")\
            .mode("overwrite")\
            .saveAsTable("silver_seller_revenue")

customer_segmentation.write\
            .format("delta")\
            .mode("overwrite")\
            .saveAsTable("silver_customer_segmentation")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
